# P&ID Medallion Pipeline — Concepts Walkthrough (Bronze → Silver)

This notebook illustrates, end to end, what we have built so far: a **Bronze**
(raw, immutable ingestion) → **Silver** (parse + topology reconstruction) pipeline
for P&ID interoperability exports (DEXPI/Proteus and INGR ISO-15926 PostProc),
on local Spark + Delta Lake.

It demonstrates the key concepts:

- **Bronze** stores the source XML *as-is* — content hash, format detection, project
  code and drawing revision captured, dedup on exact bytes.
- **Silver** *re-houses* the validated `pidtool`/`bppidsys` reconstruction (the
  "crown jewel") — recovering inline valves the raw graph lacks — and emits typed
  tables: components, segments, connections, equipment.
- The **oracle firewall**: the source turnover assignment is carried as *quarantined*
  lineage, never computed on.
- The **`flow_sense`** four-state directional overlay and the **`derived`** provenance
  flag on every reified connection.
- **Format parity**: DEXPI and PostProc flow through one code path into one schema.
- A real-data finding: **`seg_tag` is not unique** (the CDC anchor-collision risk).
- **Stage D**: a declarative expectation suite writes the **`silver_quality`**
  punch list; only two structural invariants hard-fail (fail for bugs, not data).

> **Run order matters.** After any kernel restart, run the cells top-to-bottom.
> Cell 1 *must* be first — it forces the venv's Spark 3.5.1 and blocks the system
> Spark 4 at `/opt/spark`.

## 0. Environment & pinned session

Two things bite on local WSL and are handled here:

1. **Which Spark.** A system `SPARK_HOME=/opt/spark` (Spark 4) shadows the venv's
   Spark 3.5.1 and breaks Delta (`DeltaCatalog` not found). Cell 1 strips it
   *before* `pyspark` is ever imported.
2. **One catalog, one warehouse.** The Hive metastore (`metastore_db/`) holds
   *names → locations*; the warehouse (`spark-warehouse/`) holds the *data*. We
   **pin both** to fixed paths so every session sees the same tables (embedded
   Derby is single-session — don't also run a `!python -m ...` subprocess while
   this notebook's session is live).

In [1]:
# --- CELL 1 — must run FIRST (before any `import pyspark`) ---
import os, sys
os.environ.pop("SPARK_HOME", None)                       # ignore system /opt/spark (Spark 4)
os.environ["PYTHONPATH"] = os.pathsep.join(
    p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if "/opt/spark" not in p)
sys.path[:] = [p for p in sys.path if "/opt/spark" not in p]
assert "pyspark" not in sys.modules, "Restart the kernel and run THIS cell first."

from pathlib import Path

# repo_root = the ProjectData repo root (the folder containing bronze/ and silver/)
repo_root = Path.cwd()
while not (repo_root / "bronze").is_dir() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
assert (repo_root / "bronze").is_dir(), f"Open this notebook inside the ProjectData repo (cwd={Path.cwd()})"

# --- the variables for this walkthrough ---
source_sample_dir = repo_root / "sample_data"                 # committed synthetic fixtures
source_dir        = repo_root / "data/exports/projectA"       # real Project A (DEXPI)
source_dir_B      = repo_root / "data/exports/projectB"       # real Project B (PostProc)
bronze_table      = "bronze.pid_documents"                    # NAMED Bronze table (metastore)
spark_warehouse   = repo_root / "spark-warehouse"             # managed-table data
metastore_db      = repo_root / "metastore_db"                # Hive/Derby catalog

# --- metastore hygiene (run BEFORE the session, Cell 2) ---
# Embedded Derby allows ONE connection. A leftover lock from a crashed or still-open
# kernel makes a new session fail with "Unable to instantiate SessionHiveMetaStoreClient".
# Clear stale locks here; flip RESET=True for a guaranteed clean slate — the metastore
# + warehouse are a throwaway PoC store, everything is rebuilt from the XML below.
import shutil
RESET = False        # set True, re-run this cell, then run the notebook top-to-bottom
if RESET:
    shutil.rmtree(metastore_db, ignore_errors=True)
    shutil.rmtree(spark_warehouse, ignore_errors=True)
for _lck in ("db.lck", "dbex.lck"):            # release a stale Derby lock (safe if unheld)
    try:
        (metastore_db / _lck).unlink(missing_ok=True)
    except Exception:
        pass
(repo_root / "derby.log").unlink(missing_ok=True)

print("repo_root       :", repo_root)
print("bronze_table    :", bronze_table)
print("spark_warehouse :", spark_warehouse)
print("metastore_db    :", metastore_db)
print("RESET           :", RESET)

repo_root       : /home/dcamacho/dev/ProjectData
bronze_table    : bronze.pid_documents
spark_warehouse : /home/dcamacho/dev/ProjectData/spark-warehouse
metastore_db    : /home/dcamacho/dev/ProjectData/metastore_db
RESET           : False


In [2]:
# --- CELL 2 — build ONE pinned Delta+Hive session (venv Spark 3.5.1) ---
from bronze.spark_session import get_spark
spark = get_spark(extra_conf={
    "spark.sql.warehouse.dir": f"file:{spark_warehouse}",
    "spark.hadoop.javax.jdo.option.ConnectionURL":
        f"jdbc:derby:;databaseName={metastore_db};create=true",
})
import pyspark
from pyspark.sql import functions as F
print("pyspark :", pyspark.__file__)   # expect .../.venv/...  NOT /opt/spark
print("Spark   :", spark.version)      # expect 3.5.1
assert "/opt/spark" not in pyspark.__file__, "Still on system Spark 4 — restart kernel, run Cell 1 first."
assert spark.version.startswith("3.5"), f"Expected Spark 3.5.x, got {spark.version}"
print("OK — venv Spark 3.5.1, Delta + Hive ready.")

your 131072x1 screen size is bogus. expect trouble
26/09/03 20:38:39 WARN Utils: Your hostname, DC01NNCOL resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/03 20:38:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/dcamacho/dev/ProjectData/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/dcamacho/.ivy2/cache
The jars for the packages stored in: /home/dcamacho/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-83b68c59-886c-4423-8c94-d8d41018628b;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 182ms :: artifacts dl 7ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   | 

pyspark : /home/dcamacho/dev/ProjectData/.venv/lib/python3.9/site-packages/pyspark/__init__.py
Spark   : 3.5.1
OK — venv Spark 3.5.1, Delta + Hive ready.


In [3]:
# --- CELL 3 (optional) — targeted rebuild (needs a WORKING metastore) ---
# Drops the named Bronze + Silver tables and their warehouse dirs so a re-run
# starts fresh (avoids DELTA_CREATE_TABLE_WITH_NON_EMPTY_LOCATION). If the
# metastore itself is wedged ("Unable to instantiate SessionHiveMetaStoreClient"),
# this cell can't help — use Cell 1's RESET=True instead (a filesystem wipe that
# runs before the session). Everything is rebuilt from the source XML below.
import shutil
_wipe = {
    "bronze": ["pid_documents"],
    "silver": ["silver_components", "silver_segments", "silver_connections",
               "silver_equipment", "silver_quality"],
}
for db, tables in _wipe.items():
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {db}")
    for t in tables:
        spark.sql(f"DROP TABLE IF EXISTS {db}.{t}")
        shutil.rmtree(spark_warehouse / f"{db}.db" / t, ignore_errors=True)
print("clean slate ready")

26/09/03 20:38:44 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/09/03 20:38:44 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/09/03 20:38:46 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/09/03 20:38:46 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore dcamacho@127.0.1.1
26/09/03 20:38:46 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


clean slate ready


## 1. Bronze — raw, immutable ingestion

Bronze lands each source file **verbatim**, one row per distinct byte-version, with:
`content` (raw bytes), a self-describing `content_hash` (`sha256:…`), the detected
`source_format` (DEXPI vs POSTPROC), the EPC `document_number` and derived
`project_code`, and the current `drawing_revision` / `drawing_revision_date`.
It **never interprets** the network model — that's Silver's job.

We ingest into the **named** Bronze table `bronze.pid_documents` in the pinned
metastore, and every stage below reads and writes named tables in that one
catalog — so the whole notebook is consistent (no path-based side door).

In [4]:
# --- pick sources: prefer the real exports, fall back to the committed samples ---
def xmls(d): return sorted(Path(d).glob("*.xml")) if Path(d).is_dir() else []
sources = [d for d in (source_dir, source_dir_B) if xmls(d)]
if not sources:
    sources = [source_sample_dir]
for d in sources:
    print(f"{len(xmls(d)):3d} xml  in  {d}")

  5 xml  in  /home/dcamacho/dev/ProjectData/data/exports/projectB


In [5]:
# --- ingest each source folder into the SAME named Bronze table ---
from bronze.notebook import ingest_folder
for d in sources:
    summary = ingest_folder(spark, source_dir=str(d), table_name=bronze_table)
    print(d.name, "->", summary)

26/09/03 20:38:50 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`bronze`.`pid_documents` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.
26/09/03 20:38:50 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/09/03 20:38:51 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/09/03 20:38:51 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/09/03 20:38:51 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/09/03 20:38:51 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


projectB -> {'ingest_run_id': '15d58dc4-aca1-4009-9216-94d1090c402b', 'files_in_batch': 5, 'rows_inserted': 5, 'rows_skipped_already_present': 0}


In [6]:
# --- inspect Bronze: both formats, lineage columns, self-describing hash ---
bronze = spark.table(bronze_table)
print("Bronze rows:", bronze.count())
bronze.groupBy("source_format").count().show()
bronze.select("document_number", "drawing_revision", "drawing_revision_date",
              "project_code", "content_hash").show(6, False)

Bronze rows: 5
+-------------+-----+
|source_format|count|
+-------------+-----+
|     POSTPROC|    5|
+-------------+-----+

+-----------------------------+----------------+---------------------+------------+-----------------------------------------------------------------------+
|document_number              |drawing_revision|drawing_revision_date|project_code|content_hash                                                           |
+-----------------------------+----------------+---------------------+------------+-----------------------------------------------------------------------+
|216097C-A14-PID-0021-0005-001|E               |2026/05/08           |216097C     |sha256:03bc31a7d6ed930b1c286cacbe75693fdd6cefef29281f6e5860b21006d2bf06|
|216097C-A14-PID-0021-0004-001|E               |2026/05/08           |216097C     |sha256:cab99a0b87087921133e67189b8afbce33fd9723ce34b7c1f89e76695fba8dd5|
|216097C-A14-PID-0021-0001-001|E               |2026/05/08           |216097C     |sha256:5aa8

**Dedup on exact bytes.** Re-ingesting the same files lands *nothing* new —
Bronze versions files by `content_hash`, so identical bytes are skipped
(`rows_skipped_already_present`).

In [7]:
# re-ingest the first folder — expect rows_inserted: 0
print(ingest_folder(spark, source_dir=str(sources[0]), table_name=bronze_table))

{'ingest_run_id': '0731a98f-c54e-4842-9bf6-57dfb2073e23', 'files_in_batch': 5, 'rows_inserted': 0, 'rows_skipped_already_present': 5}


## 2. Silver — parse + topology reconstruction

Silver reads Bronze, picks the adapter from `source_format`, builds the DOM from
the raw bytes, and runs the **validated reconstruction** (vendored under
`silver/_recon/`, re-housed not re-derived). It emits four typed Delta tables and
carries the source turnover assignment as **quarantined** lineage.

We run it **in-session** (same notebook session) reading the named Bronze table —
so the Silver tables land in this session's pinned catalog and are queryable by
name.

In [8]:
# --- run Silver Stage A+B in-session ---
from silver.notebook import reconstruct
counts = reconstruct(spark, bronze_table=bronze_table, silver_schema="silver")
print(counts)

26/09/03 20:39:11 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`silver`.`silver_components` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.
26/09/03 20:39:12 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`silver`.`silver_segments` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.
26/09/03 20:39:13 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`silver`.`silver_connections` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.
26/09/03 20:39:14 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`silver`.`silver_eq

{'silver_components': 1987, 'silver_segments': 807, 'silver_connections': 1396, 'silver_equipment': 9}


In [9]:
for t in ["silver_components", "silver_segments", "silver_connections", "silver_equipment"]:
    print(f"{t:22s} {spark.table('silver.' + t).count():6d} rows")

silver_components        1987 rows
silver_segments           807 rows
silver_connections       1396 rows
silver_equipment            9 rows


## 3. The concepts, illustrated in the data

### 3a. The crown jewel — inline valves recovered

The raw `<Connection>` records wire only each segment's two endpoints; inline valves
are missing. The reconstruction repairs the topology and re-inserts them. Here they
appear as real components flagged `is_valve` — in **both** formats.

In [10]:
spark.table("silver.silver_components") \
     .groupBy("source_format", "is_valve").count() \
     .orderBy("source_format", "is_valve").show()

+-------------+--------+-----+
|source_format|is_valve|count|
+-------------+--------+-----+
|     POSTPROC|   false| 1755|
|     POSTPROC|    true|  232|
+-------------+--------+-----+



### 3b. The oracle firewall

`src_turnover` / `src_subsystem` (the source commissioning assignment) is **carried**
on the segment row — but it sits on its own columns and **nothing computes on it**.
It is the validation *answer key*, quarantined so the ~97% agreement stays honest.

In [11]:
spark.table("silver.silver_segments") \
     .select("seg_tag", "fluid", "piping_materials_class",
             "src_turnover", "src_subsystem", "project_code").show(6, False)

+------------------------+-----+----------------------+------------+-------------+------------+
|seg_tag                 |fluid|piping_materials_class|src_turnover|src_subsystem|project_code|
+------------------------+-----+----------------------+------------+-------------+------------+
|3"-PG-1419202-F400H-H   |PG   |F400H                 |NULL        |NULL         |216097C     |
|4"-PG-1419202-F400H-H   |PG   |F400H                 |NULL        |NULL         |216097C     |
|8"-PG-1419202-F400H-H   |PG   |F400H                 |NULL        |NULL         |216097C     |
|8"-PG-1419202-F400H-H   |PG   |F400H                 |NULL        |NULL         |216097C     |
|3/4"-PG-1419105-D341HD-H|PG   |D341HD                |NULL        |NULL         |216097C     |
|44"-PG-1419105-D341HD-H |PG   |D341HD                |NULL        |NULL         |216097C     |
+------------------------+-----+----------------------+------------+-------------+------------+
only showing top 6 rows



### 3c. `flow_sense` — the four-state directional overlay

Direction is a *separate overlay* on the undirected connection, and it has four
states — `none` and `both` are real and a boolean couldn't hold them. Both formats
produce all four.

In [12]:
spark.table("silver.silver_connections") \
     .groupBy("source_format", "flow_sense").count() \
     .orderBy("source_format", "flow_sense").show()

+-------------+----------+-----+
|source_format|flow_sense|count|
+-------------+----------+-----+
|     POSTPROC|      both|    9|
|     POSTPROC|   forward|  616|
|     POSTPROC|      none|  135|
|     POSTPROC|   reverse|  636|
+-------------+----------+-----+



### 3d. `derived` — Source (stated) vs Derived (reconstructed) edges

Every reified connection carries provenance: `derived=false` where the edge was
stated in a source `<Connection>`, `derived=true` where the reconstruction inferred
it. This is what keeps the semantic layer from asserting inferred topology as fact.

In [13]:
spark.table("silver.silver_connections").groupBy("source_format", "derived").count().show()

+-------------+-------+-----+
|source_format|derived|count|
+-------------+-------+-----+
|     POSTPROC|   true|  726|
|     POSTPROC|  false|  670|
+-------------+-------+-----+



### 3e. Format parity — two standards, one schema

DEXPI and PostProc coexist in the same tables with identical columns — the
interoperability promise made concrete.

In [14]:
spark.table("silver.silver_segments").groupBy("source_format").count().show()

+-------------+-----+
|source_format|count|
+-------------+-----+
|     POSTPROC|  807|
+-------------+-----+



### 3f. Real-data finding — `seg_tag` is not unique

Distinct `segment_id`s can compose to the **same** business `seg_tag`. So the
composed tag cannot stand alone as the CDC segment anchor — it needs a
disambiguator, and the quality gate owes an *anchor-collision* flag. (This is why
we recorded it in the spec's §3.5.)

In [15]:
(spark.table("silver.silver_segments")
   .groupBy("seg_tag").count().filter("count > 1")
   .orderBy(F.desc("count")).show(10, False))

+------------------------+-----+
|seg_tag                 |count|
+------------------------+-----+
|Conn to process/supply- |37   |
|36"-PG-1417205-D24P1HD-H|24   |
|36"-PG-1418103-D341HD-H |22   |
|36"-PG-1415101-D341H-H  |15   |
|PG-1418103-D341HD-H     |14   |
|36"-PG-1417202-D24P1HD-H|12   |
|28"-PG-1416201-F341HD-H |12   |
|40"-PG-1416304-D341HD-H |12   |
|36"-PG-1415109-D341H-H  |10   |
|44"-PG-1419105-D341HD-H |9    |
+------------------------+-----+
only showing top 10 rows



## 3g. Stage D — the data-quality punch list

Stage D promotes the specs' advisory flags to a **declarative expectation suite**
(rules-as-data, `silver/quality_suite.py`) and writes `silver_quality` — the
per-drawing / per-project **punch list** a pre-commissioning engineer fixes at
source *before* systemization runs (segments missing fluid / piping-class /
diameter, tags that break the naming convention, the `seg_tag` anchor-collision,
prefix-integrity, orphans). The gate is **observe-and-record**: everything flags
and flows. Only two *structural invariants* — an **oracle leak** (§5) or an
**unflagged `derived` edge** (§4) — hard-fail, because those are pipeline bugs,
not dirty data.

It also denormalises a `quality_gate` enum (`clean`/`flagged`/`quarantined`) back
onto every object row, so a cautious consumer can filter without joining the
ledger.

In [16]:
# --- run Stage D in-session; it reads the four Silver tables ---
from silver.notebook import quality
# refdata_path lights up the reference-backed checks (unknown fluid/unit, naming);
# without it those skip cleanly. Point it at the project's Reference_Data.xlsx:
refdata_path = repo_root / "Reference_Data.xlsx"
summary = quality(spark, refdata_path=str(refdata_path) if refdata_path.exists() else None)
import json; print(json.dumps(summary, indent=2, default=str))

26/09/03 20:39:21 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`silver`.`silver_quality` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.


{
  "expectations_run": 14,
  "expectations_skipped": 0,
  "ledger_rows": 1285,
  "hard_failures": 0,
  "objects_flagged": 1463,
  "objects_quarantined": 0,
  "run_warnings": 0,
  "warnings": [],
  "skipped": [],
  "quality_table": "silver.silver_quality"
}


**The punch list** — one row per flag occurrence, ordered worst-first. This
is the artefact the engineer works from.

In [17]:
from pyspark.sql import functions as F
sev_rank = F.when(F.col("severity") == "error", 0).when(F.col("severity") == "warn", 1).otherwise(2)
(spark.table("silver.silver_quality")
   .withColumn("_r", sev_rank)
   .orderBy("_r", "flag")
   .select("severity", "gate", "object_kind", "flag", "drawing_number", "detail")
   .show(40, False))

+--------+----+-----------+---------------------------+-----------------------------+---------------------------------------------------------------------------------------------------------------+
|severity|gate|object_kind|flag                       |drawing_number               |detail                                                                                                         |
+--------+----+-----------+---------------------------+-----------------------------+---------------------------------------------------------------------------------------------------------------+
|warn    |flag|component  |instrument_tag_noncompliant|216097C-A14-PID-0021-0005-001|instrument tag '14-N2-12867' does not match the project instrument naming pattern                              |
|warn    |flag|component  |instrument_tag_noncompliant|216097C-A14-PID-0021-0005-001|instrument tag '14-H2-12866' does not match the project instrument naming pattern                              |
|warn    |

**Punch-list rollup by flag** — where the data gaps concentrate.

In [18]:
(spark.table("silver.silver_quality")
   .groupBy("flag", "severity", "gate").count()
   .orderBy(F.desc("count")).show(30, False))

+----------------------------+--------+----+-----+
|flag                        |severity|gate|count|
+----------------------------+--------+----+-----+
|orphan_component            |info    |flag|619  |
|segment_missing_diameter    |warn    |flag|157  |
|seg_tag_anchor_collision    |info    |flag|150  |
|segment_insulation_absent   |info    |flag|108  |
|segment_missing_piping_class|warn    |flag|104  |
|segment_missing_fluid       |warn    |flag|103  |
|instrument_tag_noncompliant |warn    |flag|33   |
|prefix_integrity            |warn    |flag|9    |
|segment_unknown_insulation  |warn    |flag|2    |
+----------------------------+--------+----+-----+



**The gate rollup on the objects themselves** — `flagged` rows still flow to
Gold and the rules; a strict consumer can exclude `quarantined` without a join.

In [19]:
for t in ["silver_segments", "silver_components"]:
    print(t)
    spark.table("silver." + t).groupBy("quality_gate").count().orderBy("quality_gate").show()

silver_segments
+------------+-----+
|quality_gate|count|
+------------+-----+
|       clean|  117|
|     flagged|  690|
+------------+-----+

silver_components
+------------+-----+
|quality_gate|count|
+------------+-----+
|       clean| 1358|
|     flagged|  629|
+------------+-----+



## 3h. Lineage trace — one attribute, Bronze bytes → Silver column

The whole point of carrying `bronze_id` / `content_hash` on every Silver row (§4)
is that any value traces back to the exact source bytes it came from. Here we
follow **insulation** end to end: the Silver `insul_purpose` column, the raw
`InsulPurpose` attribute pulled straight out of the Bronze XML, and the `seg_tag`
suffix (e.g. `-H`) are the *same source fact reached three ways*. Reading it from
the segment's own `<GenericAttributes>` block mirrors `pidsys.master_data.ga()`
exactly. The identical three-hop walk traces fluid, diameter, piping class, or the
quarantined oracle columns — insulation isn't special.

In [20]:
# --- Lineage trace: Insulation from Bronze bytes -> Silver columns ---
import xml.etree.ElementTree as ET

seg = spark.table("silver.silver_segments")

# 1) the Silver insulation columns + the lineage keys that trace each row to source
(seg.select("segment_id", "seg_tag", "insul_purpose", "insul_type", "insul_thick",
            "bronze_id", "content_hash", "drawing_number")
    .where("insul_purpose is not null")
    .show(8, False))

# 2) pull InsulPurpose straight out of the raw Bronze XML for one segment and compare.
bronze = spark.table(bronze_table)

row = (seg.where("insul_purpose is not null")
          .join(bronze.select("bronze_id", "content"), "bronze_id")
          .select("segment_id", "insul_purpose", "insul_type", "insul_thick", "content")
          .head())

def _ln(el):                                   # strip XML namespace
    return el.tag.split("}")[-1]

def insul_from_bytes(content, seg_id):
    # mirror pidsys.master_data.ga(): the segment's OWN <GenericAttributes> block
    root = ET.fromstring(bytes(content))
    for el in root.iter():
        if _ln(el) == "PipingNetworkSegment" and el.get("ID") == seg_id:
            return {g.get("Name"): g.get("Value")
                    for gas in el if _ln(gas) == "GenericAttributes"
                    for g in gas if _ln(g) == "GenericAttribute"
                    and (g.get("Name") or "").startswith("Insul")}
    return {}

if row is None:
    print("no segment with a non-null insul_purpose yet — run Silver Stage A+B first")
else:
    tag = seg.where(seg.segment_id == row.segment_id).head().seg_tag
    print("segment_id :", row.segment_id)
    print("seg_tag    :", tag, "  (last token = insulation purpose)")
    print("SILVER cols:", dict(insul_purpose=row.insul_purpose,
                               insul_type=row.insul_type, insul_thick=row.insul_thick))
    print("BRONZE XML :", insul_from_bytes(row.content, row.segment_id))
    # to trace a SPECIFIC flagged segment: replace the filter in `row` with
    #   .where("segment_id = '<the id from silver_quality.object_id>'")

+----------------------------------+------------------------+-------------+----------+-----------+------------------------------------+-----------------------------------------------------------------------+-----------------------------+
|segment_id                        |seg_tag                 |insul_purpose|insul_type|insul_thick|bronze_id                           |content_hash                                                           |drawing_number               |
+----------------------------------+------------------------+-------------+----------+-----------+------------------------------------+-----------------------------------------------------------------------+-----------------------------+
|SGC236C79443624431BC3EE7BFB0BF64C5|28"-PG-1418103-D341HD-H |H            |NULL      |NULL       |86b9f7b3-c100-4b46-b9d6-289a5fa44919|sha256:cab99a0b87087921133e67189b8afbce33fd9723ce34b7c1f89e76695fba8dd5|216097C-A14-PID-0021-0004-001|
|SGA0538D7989924F7AAB36AC7432CF7A13|2"-PG-141810

> If `insul_purpose` comes back all-null in Silver while the `seg_tag` still
> shows a `-H`/`-N` suffix, that mismatch *is* the finding — the value reached the
> composed tag but the column extraction missed it (Bronze→Silver drift), which is
> exactly what this trace is built to catch.

## 3i. Stage C — joining the P&IDs (off-page connectors)

Stage B reconstructs each drawing on its own; **Stage C** joins them into one
plant by matching **off-page connectors** (OPCs) across sheets — by `OPCTag` for
PostProc, by GUID for DEXPI (`bppidsys.offpage.match_pairs`, re-housed). Each
matched pair becomes one undirected, always-`derived` **`OffPage`** edge in
`silver_connections` that spans two drawings; an OPC whose mate isn't in the
loaded set is an **open boundary** — the system continues off-sheet — recorded as
an `opc_open_boundary` flag, never dropped. Run it after reconstruct().

In [21]:
# --- run Stage C in-session (harvest OPCs per sheet -> match across sheets) ---
from silver.notebook import assemble
import json
stats = assemble(spark, bronze_table=bronze_table, silver_schema="silver")
print(json.dumps(stats, indent=2, default=str))

{
  "opc_records": 42,
  "opc_stitched": 6,
  "opc_offset": 30,
  "silver_connections": "silver.silver_connections"
}


**The cross-document edges** — one row per stitched OPC pair, `derived=true`,
joining two drawings into one connectivity graph.

In [22]:
off = spark.table("silver.silver_connections").where("conn_type = 'OffPage'")
print("OffPage edges:", off.count())
off.select("connection_id", "from_id", "to_id", "derived", "flow_sense").show(20, False)

OffPage edges: 6
+-----------------------------------------------------------------------+----------------------------------+----------------------------------+-------+----------+
|connection_id                                                          |from_id                           |to_id                             |derived|flow_sense|
+-----------------------------------------------------------------------+----------------------------------+----------------------------------+-------+----------+
|sha256:31ce937b8110469b29fef94073a9c1d6d1f3a5b0424682c0128060bccad934b4|SP48F67504784E42AB811FE4BC937E8518|SP7BF76D6845964F67A70844B976F6E63D|true   |none      |
|sha256:56659c0999da7abb4d6764b14674ea1e59d3fcfb19b6dcbe4f52b2b8196f7d82|SP5A970EF3697544E992295BEC0CC945D1|SP647F24E0C0AF4082A76710FD640AD236|true   |none      |
|sha256:28b9d3cc5b41ba3e513eeb7f985480e0dc110dc023df13bfaf705a019d6ebae5|SP0A0E837B63934881BB03245C7E8B16C2|SPF63D5E8885224665B32107A16A5AD41A|true   |none      |
|sha2

**Open boundaries** — OPCs with no mate in the loaded set. Not errors: the
system continues onto a sheet that wasn't loaded. Load more sheets and these
resolve into `OffPage` edges.

In [23]:
(spark.table("silver.silver_quality").where("flag = 'opc_open_boundary'")
   .select("object_id", "drawing_number", "detail").show(20, False))

+----------------------------------+-----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|object_id                         |drawing_number               |detail                                                                                                                                                                      |
+----------------------------------+-----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|SP36D305B31145498A9348D9F25F0D8865|216097C-A14-PID-0021-0002-001|OPC '3800' on drawing 216097C-A14-PID-0021-0002-001 (paired drawing A35-0056-001) has no mate in the loaded set — open boundary (the system continues off-set; not an error)|
|SP908D02FF882746AD9C16A09492778244|2160

## 3j. Stage E — change data capture, a two-revision narrative

SmartPlant re-exports the **whole** drawing XML for one symbol move, and re-mints
an element's UID when it is deleted and recreated — so a file hash (or the UID)
marks everything changed. **Stage E** answers the real questions with an
*anchor-match* identity that survives delete+recreate (equipment tag / composed
seg tag / `(segment, class)` bucket) and three separated hashes: `anchor_hash`
(no UID), `content_hash_eng` (engineering attrs + neighbour **anchor** sets — the
Modify trigger), and `content_hash_audit` (adds UID + the quarantined oracle, so a
recreate is *visible* but stays *inert* for engineering CDC). It diffs, per
drawing, the two most recent Bronze versions and writes New/Modified/Deleted — the
interval open/close events Gold consumes.

To make that concrete we run a **real EPC event**: a Project-B Unit-22 (Steam &
BFW) set is **issued at Rev C for HAZOP**, then **re-issued at Rev D for design**.
Between the two, engineering changed some lines / valves / instruments / equipment
— and *every element UID is re-minted*. We ingest both revisions as two Bronze
versions and let Stage E find the real change. (Runs on its **own** `silver_cdc_demo`
schema so the main walkthrough above is untouched.)

In [24]:
# --- generate the two-revision narrative (4 synthetic PostProc XMLs) ---
from silver.demo_cdc import write_narrative, D1, D2
paths = write_narrative(str(repo_root / "_cdc_demo"))
print("Rev C:", paths["rev1_C"]); print("Rev D:", paths["rev2_D"])

Rev C: /home/dcamacho/dev/ProjectData/_cdc_demo/rev1_C
Rev D: /home/dcamacho/dev/ProjectData/_cdc_demo/rev2_D


In [25]:
# --- run the mini medallion on an ISOLATED demo schema ---
import shutil, json
from bronze.notebook import ingest_folder
from silver.notebook import reconstruct, changes

demo_bronze, demo_schema = "bronze.cdc_demo", "silver_cdc_demo"
# clean any prior demo run (tables + warehouse dirs)
spark.sql(f"DROP TABLE IF EXISTS {demo_bronze}")
shutil.rmtree(spark_warehouse / "bronze.db" / "cdc_demo", ignore_errors=True)
for t in ["silver_components","silver_segments","silver_connections","silver_equipment","silver_cdc"]:
    spark.sql(f"DROP TABLE IF EXISTS {demo_schema}.{t}")
    shutil.rmtree(spark_warehouse / f"{demo_schema}.db" / t, ignore_errors=True)

# Rev C issued -> ingest + reconstruct (the plant as HAZOP saw it)
ingest_folder(spark, source_dir=paths["rev1_C"], table_name=demo_bronze)
reconstruct(spark, bronze_table=demo_bronze, silver_schema=demo_schema)
# Rev D re-issued -> append the second version, reconstruct again
ingest_folder(spark, source_dir=paths["rev2_D"], table_name=demo_bronze)
reconstruct(spark, bronze_table=demo_bronze, silver_schema=demo_schema)

# Stage E: the change report
summary = changes(spark, bronze_table=demo_bronze, silver_schema=demo_schema)
print(json.dumps(summary, indent=2, default=str))   # expect 2 drawings, ~15 deltas

26/09/03 20:39:38 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`bronze`.`cdc_demo` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.
26/09/03 20:39:41 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`silver_cdc_demo`.`silver_components` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.
26/09/03 20:39:42 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`silver_cdc_demo`.`silver_segments` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.
26/09/03 20:39:43 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`silver_cdc

{
  "drawings_with_two_versions": 2,
  "deltas": 15,
  "New": 8,
  "Modified": 3,
  "Deleted": 4,
  "total": 15,
  "silver_cdc": "silver_cdc_demo.silver_cdc"
}


26/09/03 20:39:53 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`silver_cdc_demo`.`silver_cdc` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.


**The change report** — grouped, then in detail. Every UID changed, yet only
the real engineering changes surface (`change_type` is the Gold interval event).

In [26]:
cdc = spark.table(f"{demo_schema}.silver_cdc")
print("total deltas:", cdc.count())
cdc.groupBy("grain", "change_type").count().orderBy("grain", "change_type").show()
(cdc.select("grain", "change_type", "drawing_number", "anchor", "detail")
    .orderBy("grain", "change_type").show(60, False))

total deltas: 15
+---------+-----------+-----+
|    grain|change_type|count|
+---------+-----------+-----+
|component|    Deleted|    3|
|component|   Modified|    1|
|component|        New|    5|
|equipment|   Modified|    1|
|equipment|        New|    1|
|  segment|    Deleted|    1|
|  segment|   Modified|    1|
|  segment|        New|    2|
+---------+-----------+-----+

+---------+-----------+-----------------------------+----------------------------------------------------------------------------------+-------------------------------------+
|grain    |change_type|drawing_number               |anchor                                                                            |detail                               |
+---------+-----------+-----------------------------+----------------------------------------------------------------------------------+-------------------------------------+
|component|Deleted    |216097C-A22-PID-0021-0015-001|CMP|SEG|216097C-A22-PID-0021-0015-001|3/4"-W

**Churn is invisible.** Every one of the ~30 elements was re-exported with a
new UID, but the re-drawn-yet-unchanged objects (line `WBF-2215103`, pump
`P-2201A`, both lines on Drawing 0016, drum `V-2202`) produce **no delta**. The
whole of Drawing 0016 was re-issued yet only its one new PSV vent line is flagged —
no engineer has to eyeball a re-issued sheet to find what moved.

Each delta is a **Gold** interval event: *New* opens `[validFrom, ∞)`, *Deleted*
closes the prior interval, *Modified* closes the old and opens the new — giving a
defensible "current truth" *and* a Rev-C-vs-Rev-D timeline, and scoping the
change-driven work (MTO delta, the new `PSV-2201` ITR, the drum-nozzle interface,
the redlined check valve) to exactly what changed. Full manifest by discipline:
`narrative_project_b/NARRATIVE.md`.

## 4. Recap

**Built (Phase-1 + Stages C, D, E):** Bronze (raw, immutable, dedup,
format-tagged) → Silver (parse + reconstruction, four typed tables) → **Stage C
assembly** (`OffPage` edges + open boundaries) → **Stage D quality gate**
(`silver_quality` punch list + `quality_gate` rollup) → **Stage E CDC**
(`silver_cdc` object-grain deltas, delete+recreate-safe), validated on real
Project A **and** Project B. Silver is complete.

**Concepts shown:** store-as-is + content hash; format detection; the reconstruction
recovering inline valves; the oracle firewall; the `flow_sense` enum and `derived`
provenance; format parity; the `seg_tag` anchor-collision; and the Stage-D
punch list with its fail-for-bugs-not-data gate policy.

**Runtime lessons baked in:** force the venv's Spark 3.5.1 (Cell 1); pin the metastore
+ warehouse; run everything **in-session** against the named tables in that one
metastore — no path-based side door, no `!python -m …` subprocess against a live Derby.

**Next:** the **Gold layer** — bi-temporal `validFrom`/`validTo` intervals over
Stage E's deltas, then the RDF/IDO projection and the Jena rule packages
(systemization, Test Packages). Stage D's reference-backed checks light up as soon
as a project `Reference_Data.xlsx` — with `Naming` and `Insulation` sheets — is
supplied.

## 5. Gold -- bi-temporal versioning over Stage E's `silver_cdc`

> **These cells are new and unexecuted.** They were written against the real
> `silver_cdc` schema (`spark.table(...).printSchema()`, confirmed against
> this notebook's own `silver_cdc_demo` output), but there is no Spark
> session or Project A/B data available where they were written -- run them
> top-to-bottom, continuing from §3j above, once `gold/` (delivered
> separately -- `gold_layer.zip` / `gold_layer/README.md` "Merging into the
> `ProjectData` repo") is on this repo's Python path.

`silver_cdc` carries **identity + change classification + hashes**, but not
the object's own engineering attribute values, and its `old_revision` /
`new_revision` are letter codes (`C`/`D`), not dates. So Gold's bi-temporal
layer over it needs two small bridges the cells below build explicitly
rather than assume:

1. **valid_from** is resolved back through **Bronze** (`bronze_layer_spec.md`
   §3.1/§6 -- Bronze is the layer that actually captured the revision issue
   date). This takes the *latest ingested revision per drawing* as the run's
   single `valid_from` -- the same one-drawing-one-origin simplification
   `gold_job.py`'s own `_valid_from_for_kind()` already documents.
2. **attrs** (the actual engineering values a Gold row carries) are fetched
   from the *current* Silver snapshot table for that grain
   (`silver_components` / `silver_segments` / `silver_equipment` /
   `silver_connections`), keyed on Stage E's own `new_uid` -- which is
   exactly that table's `{grain}_id` column, since Stage E's anchor-match
   identity re-mints the internal id every version but the anchor stays
   stable (`silver_layer_spec.md` §3.5).

`gold/temporal.py::apply_delta` is unchanged from the tested package
(`gold_layer/tests/`, 60/60 green). The application loop below uses
`apply_silver_cdc_events_tolerant`, not the strict `apply_silver_cdc_events`:
a real `silver_cdc` batch's `anchor` is a **bucket key** for components
(`silver_layer_spec.md` §3.5 pairs same-class siblings on one segment within
a shared bucket, not a per-instance string), so one batch can legitimately
carry two simultaneous events for one anchor -- the same non-uniqueness
§3f already documents for `seg_tag`, one layer up. The tolerant path
collects each such collision as a `CdcAnomaly` (Stage D's `silver_quality`
"flag, don't crash" precedent) instead of aborting the whole run on the
first one.


In [ ]:
import sys
from pathlib import Path

# gold/ ships separately (gold_layer.zip) -- point this at wherever it was
# merged into the repo. Adjust if it doesn't land at gold_layer/ next to
# bronze/ and silver/ (see gold_layer/README.md).
gold_layer_root = repo_root
if not (gold_layer_root / "gold").exists():
    raise RuntimeError(
        f"gold/ package not found at {gold_layer_root} -- merge gold_layer.zip's "
        "gold/ into the repo first (gold_layer/README.md 'Merging into the ProjectData repo')."
    )
sys.path.insert(0, str(gold_layer_root))

import gold
print("gold package loaded from:", Path(gold.__file__).parent)


RuntimeError: gold/ package not found at /home/dcamacho/dev/ProjectData/gold_layer -- merge gold_layer.zip's gold/ into the repo first (gold_layer/README.md 'Merging into the ProjectData repo').

In [ ]:
# --- Bridge 1: resolve a real valid_from date, per drawing, from Bronze ---
# silver_cdc's old_revision/new_revision are letter codes (e.g. 'C'/'D'), not
# dates -- bronze_layer_spec.md §6 is where the real, verbatim issue date
# lives (drawing_revision_date), keyed on document_number == silver_cdc's
# drawing_number.
from datetime import datetime, date
from pyspark.sql import functions as F

# Extend this if a project's Reference_Data / export uses a format not listed
# (bronze_layer_spec.md §6: verbatim, project-scoped -- DDMMMYY for DEXPI/A,
# YYYY/MM/DD for PostProc/B are the two the spec names; add others as seen).
DATE_FORMATS = ("%Y-%m-%d", "%Y/%m/%d", "%d%b%y", "%d-%b-%y", "%d/%m/%Y", "%m/%d/%Y")


def _parse_revision_date(raw: str) -> date:
    for fmt in DATE_FORMATS:
        try:
            return datetime.strptime(str(raw).strip(), fmt).date()
        except ValueError:
            continue
    raise ValueError(
        f"drawing_revision_date {raw!r} doesn't match any of {DATE_FORMATS} -- "
        "add this project's format above."
    )


def resolve_drawing_valid_from(spark, bronze_table: str) -> dict:
    """{document_number: latest known drawing_revision_date} -- the run's
    single valid_from per drawing (see §5's markdown for why one date, not
    per-row)."""
    rows = (spark.table(bronze_table)
                 .select("document_number", "drawing_revision_date")
                 .where(F.col("drawing_revision_date").isNotNull())
                 .distinct()
                 .toPandas())
    out = {}
    for doc, raw in zip(rows["document_number"], rows["drawing_revision_date"]):
        d = _parse_revision_date(raw)
        if doc not in out or d > out[doc]:
            out[doc] = d
    return out


def resolve_all_revision_dates(spark, bronze_table: str) -> dict:
    """{(document_number, drawing_revision): date} -- every revision seen,
    for point-in-time ("as of Rev C") demonstrations below, not just the
    latest."""
    rows = (spark.table(bronze_table)
                 .select("document_number", "drawing_revision", "drawing_revision_date")
                 .where(F.col("drawing_revision_date").isNotNull())
                 .distinct()
                 .toPandas())
    return {
        (doc, rev): _parse_revision_date(raw)
        for doc, rev, raw in zip(rows["document_number"], rows["drawing_revision"], rows["drawing_revision_date"])
    }


In [ ]:
# --- Bridge 2: silver_cdc row -> gold.silver_cdc.SilverCdcEvent ---
from gold.silver_cdc import SilverCdcEvent, apply_silver_cdc_events_tolerant
from gold.temporal import DeltaType, current_truth

GRAIN_TABLE = {"component": "silver_components", "segment": "silver_segments",
               "equipment": "silver_equipment", "connection": "silver_connections"}
GRAIN_ID_COL = {"component": "component_id", "segment": "segment_id",
                 "equipment": "equipment_id", "connection": "connection_id"}


def cdc_to_gold_events(spark, cdc_df, silver_schema: str, bronze_table: str) -> list:
    valid_from_by_doc = resolve_drawing_valid_from(spark, bronze_table)
    cdc_pd = cdc_df.toPandas()
    if cdc_pd.empty:
        return []

    # one pandas frame per grain actually present, so New/Modified rows can
    # look their attrs up by Stage E's new_uid (== that table's *_id column)
    # without round-tripping to Spark per row.
    attrs_by_grain = {
        g: spark.table(f"{silver_schema}.{t}").toPandas().set_index(GRAIN_ID_COL[g])
        for g, t in GRAIN_TABLE.items()
        if g in set(cdc_pd["grain"])
    }

    events = []
    skipped = 0
    for row in cdc_pd.itertuples():
        valid_from = valid_from_by_doc.get(row.drawing_number)
        if valid_from is None:
            print(f"skip {row.cdc_id}: no Bronze drawing_revision_date for drawing {row.drawing_number!r}")
            skipped += 1
            continue

        if row.change_type == "Deleted":
            attrs = {}
        else:
            table = attrs_by_grain.get(row.grain)
            if table is None or row.new_uid not in table.index:
                print(f"skip {row.cdc_id}: new_uid {row.new_uid!r} not found in "
                      f"{silver_schema}.{GRAIN_TABLE.get(row.grain)}")
                skipped += 1
                continue
            attrs = table.loc[row.new_uid].to_dict()

        tx = row.transaction_ts
        tx = tx.to_pydatetime() if hasattr(tx, "to_pydatetime") else tx

        events.append(SilverCdcEvent(
            object_kind=row.grain,
            anchor_id=row.anchor,
            delta_type=DeltaType(row.change_type),
            drawing_number=row.drawing_number,
            drawing_revision_date=valid_from,
            bronze_ingested_at=tx,
            attrs=attrs,
            content_hash_eng=getattr(row, "new_content_hash_eng", None) or getattr(row, "old_content_hash_eng", None),
            content_hash_audit=getattr(row, "new_content_hash_audit", None) or getattr(row, "old_content_hash_audit", None),
        ))

    if skipped:
        print(f"{skipped} of {len(cdc_pd)} silver_cdc rows skipped (see reasons above)")
    return events


### 5a. Running it on the Stage E Rev C -> Rev D narrative (§3j)

The demo schema already has a real `silver_cdc` from §3j's HAZOP -> design
re-issue narrative -- the natural first target.


In [ ]:
# de-dupe exact repeat rows first (a re-run of §3j's cells before clearing
# the demo schema appends silver_cdc again rather than overwriting it, so a
# kernel restart mid-walkthrough is the other common way to see duplicates
# here, not just a genuine bucket collision).
demo_cdc_df = spark.table(f"{demo_schema}.silver_cdc").dropDuplicates(["cdc_id"])
demo_events = cdc_to_gold_events(spark, demo_cdc_df, silver_schema=demo_schema, bronze_table=demo_bronze)
print(f"{len(demo_events)} Gold events resolved from {demo_cdc_df.count()} silver_cdc rows")

gold_rows, anomalies = apply_silver_cdc_events_tolerant({}, demo_events)
for kind, rows in gold_rows.items():
    print(f"  {kind:<10} {len(rows):3d} row-versions, {len(current_truth(rows)):3d} current")

if anomalies:
    print(f"\n{len(anomalies)} anomalies (this run's Gold punch list -- not a crash):")
    for a in anomalies:
        print(f"  [{a.reason}] {a.event.object_kind} {a.event.anchor_id!r}: {a.detail}")


**A concrete bi-temporal query.** Pick whichever anchor actually got Modified
in this run (any anchor with more than one row-version) and query both time
axes independently on it -- current truth vs. what was true during its
*first* validity window.


In [ ]:
demo_kind = demo_anchor = demo_history = None
for kind, rows in gold_rows.items():
    by_anchor = {}
    for r in rows:
        by_anchor.setdefault(r.anchor_id, []).append(r)
    for anchor, hist in by_anchor.items():
        if len(hist) > 1:
            demo_kind, demo_anchor, demo_history = kind, anchor, sorted(hist, key=lambda r: r.tx_from)
            break
    if demo_history:
        break

if demo_history is None:
    print("no anchor in this run has more than one row-version -- nothing to demonstrate a supersession with")
else:
    print(f"history for {demo_kind} {demo_anchor!r}:")
    for r in demo_history:
        print(f"  valid=[{r.valid_from}, {r.valid_to}) tx=[{r.tx_from}, {r.tx_to})  superseded_by={r.superseded_by_delta}")
    print()
    print("current truth attrs                    :", current_truth(demo_history)[0].attrs if current_truth(demo_history) else None)
    print("truth as of the FIRST row's own validity:",
          [r.attrs for r in current_truth(demo_history, as_of_valid=demo_history[0].valid_from)])


### 5b. Running it on the main Project A/B walkthrough

Stage E needs at least two ingested Bronze versions of a drawing to diff
(§3j's own framing). The main walkthrough above only ingested one version
per drawing, so this will legitimately produce nothing (or fail cleanly) the
first time -- ingest a second revision of the same drawing(s) into
`bronze_table` to exercise this for real. Left in as the natural next call,
not skipped, so the notebook already shows how the main path plugs in.


In [ ]:
from silver.notebook import changes
import json

try:
    main_cdc_summary = changes(spark, bronze_table=bronze_table, silver_schema="silver")
    print(json.dumps(main_cdc_summary, indent=2, default=str))
    main_cdc_df = spark.table("silver.silver_cdc").dropDuplicates(["cdc_id"])
    main_events = cdc_to_gold_events(spark, main_cdc_df, silver_schema="silver", bronze_table=bronze_table)
    main_gold_rows, main_anomalies = apply_silver_cdc_events_tolerant({}, main_events)
    for kind, rows in main_gold_rows.items():
        print(f"  {kind:<10} {len(rows):3d} row-versions, {len(current_truth(rows)):3d} current")
    if main_anomalies:
        print(f"{len(main_anomalies)} anomalies -- see §5a's cell above for the same reporting pattern")
except Exception as e:
    print(f"Stage E on the main schema needs a second Bronze revision to diff against; "
          f"skipping for now ({e}). §5a above already exercises the full path on the demo narrative.")


## 6. Gold -- the RDF/IDO projection over the current Silver snapshot

`gold/rdf_mapper.py` was written directly against `silver_layer_spec.md`
§4's table family, so it maps onto `silver_components` / `silver_segments` /
`silver_equipment` / `silver_connections` as-is (one known simplification:
`map_connection` addresses every endpoint as a component, so a `Nozzle`-type
edge's endpoint resolves to a component-shaped URI even where the true
endpoint is a nozzle -- fine for this walkthrough, worth tightening before
this feeds a real systemization run).


In [ ]:
segments    = spark.table("silver.silver_segments").toPandas().to_dict("records")
components  = spark.table("silver.silver_components").toPandas().to_dict("records")
equipment   = spark.table("silver.silver_equipment").toPandas().to_dict("records")
connections = spark.table("silver.silver_connections").toPandas().to_dict("records")
print(f"{len(components)} components, {len(segments)} segments, "
      f"{len(equipment)} equipment, {len(connections)} connections")


In [ ]:
# graph:refdata source -- the project's own Reference_Data.xlsx (already used
# by Stage D above, §3g). Column names are printed so the renames below can
# be corrected to match this project's actual headers if they differ.
import pandas as pd

if refdata_path.exists():
    fluid_sheet = pd.read_excel(refdata_path, sheet_name="Fluid")
    boundary_sheet = pd.read_excel(refdata_path, sheet_name="Boundary")
    print("Fluid sheet columns   :", list(fluid_sheet.columns))
    print("Boundary sheet columns:", list(boundary_sheet.columns))

    # ADJUST these renames if the printed columns above differ.
    fluid_catalogue = fluid_sheet.rename(columns={
        "FluidCode": "fluid_code", "Category": "category", "Subcategory": "subcategory",
    })[["fluid_code", "category", "subcategory"]].to_dict("records")
    boundary_rows = boundary_sheet.rename(columns={
        "ComponentClass": "component_class", "Role": "role",
    })[["component_class", "role"]].to_dict("records")
else:
    fluid_catalogue, boundary_rows = [], []
    print(f"{refdata_path} not found -- graph:refdata stays empty this run "
          "(every fluid classifies as 'utility', rules_reference.py's documented fallback)")


In [ ]:
from gold.gold_job import GoldInputs, build_rdf_dataset
from gold import vocab as v

inputs = GoldInputs(
    components=components, segments=segments, equipment=equipment, connections=connections,
    fluid_catalogue=fluid_catalogue, boundary_rows=boundary_rows,
    drawing_lineage={},  # only used by the deprecated snapshot-diff path (§5 above uses silver_cdc directly)
)
ds = build_rdf_dataset(inputs)

print(f"{len(ds)} quads across {len(ds.graphs())} named graphs:")
for g in sorted(ds.graphs()):
    print(f"  {g:<45} {sum(1 for _ in ds.triples(graph=g)):>6} quads")


### 6a. The oracle-quarantine guard, on the real projection

`silver_segments.src_turnover` / `src_subsystem` are quarantined in Silver
already (§3b above); this re-asserts the same invariant at the RDF layer --
a hard failure, not a lint warning, if a mapper ever routes them outside
`graph:oracle`.


In [ ]:
from gold.oracle_guard import assert_oracle_confined
assert_oracle_confined(ds)
print("oracle-quarantine invariant holds on the real projection: "
      "src_turnover / src_subsystem never leaked outside graph:oracle")


### 6b. Declarative fluid classification & the three directional guards, on real data

Same functions validated in `gold_layer/tests/` against the synthetic
fixture, now run against this project's actual fluid catalogue and
connectivity.


In [ ]:
from collections import Counter
from gold.rules_reference import load_fluid_catalogue, classify_fluid_category, is_self_owning

catalogue = load_fluid_catalogue(ds)
seg_fluids = [s.get("fluid") for s in segments if s.get("fluid")]
dist = Counter(classify_fluid_category(f, catalogue) for f in seg_fluids)

print("segment fluid classification across the plant:")
for cat, n in dist.most_common():
    print(f"  {cat:<16} {n:5d} segments")
print("self-owning (Flare / Steam-Condensate -- never traced to a consumer):",
      sum(1 for f in seg_fluids if is_self_owning(f, catalogue)), "/", len(seg_fluids))


In [ ]:
from gold.rules_reference import flare_guard, directional_consumer_guard, relief_attribution
from gold.rdf_model import URIRef


def comp(obj_id):
    return URIRef(v.uri(v.PIDSYS + "component/", obj_id))


skipped_flare = skipped_consumer = checked = 0
for conn in connections:
    if conn.get("flow_sense") not in ("forward", "reverse", "both"):
        continue
    checked += 1
    frm, to = comp(conn["from_id"]), comp(conn["to_id"])
    if flare_guard(ds, frm, to, catalogue):
        skipped_flare += 1
    if directional_consumer_guard(ds, frm, to):
        skipped_consumer += 1

print(f"{checked} directed connections checked")
print(f"  flare_guard would skip               : {skipped_flare}")
print(f"  directional_consumer_guard would skip: {skipped_consumer}")

relief_classes = {"SafetyValveOrFitting", "Reliefdevices"}
relief_components = [c for c in components if c.get("component_class") in relief_classes]
print(f"\n{len(relief_components)} relief-class components; attribution (protected side):")
for c in relief_components[:20]:
    protected = relief_attribution(ds, comp(c["component_id"]))
    print(f"  {c['component_id']:<14} tag={str(c.get('tag')):<14} -> {protected}")


## 7. The SPARQL query surface, on the real graph


In [ ]:
from gold.sparql_queries import EXAMPLE_QUERIES, run_local_pattern

derived_hits = list(run_local_pattern(ds, p=URIRef(v.P_DERIVED), graph=v.GRAPH_MASTERDATA))
derived_true = sum(1 for q in derived_hits if q.o.value is True)
print(f"{derived_true} / {len(derived_hits)} connections are derived "
      "(reconstruction-inferred, not source-stated)")

print()
print("the oracle cross-check -- the ONLY query allowed to join graph:results "
      "against graph:oracle, read-only validation reporting (needs graph:results "
      "from a real systemization run to return rows):")
print(EXAMPLE_QUERIES["oracle_cross_check"])


## 8. Pushing to a real Fuseki (illustrative -- no Fuseki reachable here either)


In [ ]:
from gold.fuseki_client import FusekiConfig, build_graph_store_put_request

cfg = FusekiConfig(base_url="http://localhost:3030", dataset="pidsys")
url, method, headers, data = build_graph_store_put_request(cfg, v.GRAPH_MASTERDATA, ds.to_turtle(v.GRAPH_MASTERDATA))
print(method, url)
print(headers)
print(f"{len(data)} bytes of Turtle ready to PUT")


## 9. Recap -- Gold added

**Built above (new -- §5 has now actually been run once against the real
Rev C -> Rev D narrative, §6-§8 are still unexecuted, run top-to-bottom
after §3j once `gold/` is merged in):** bi-temporal versioning consuming
Stage E's real `silver_cdc` directly (§5, via the two bridges §5 documents
rather than assumes), the RDF/IDO projection over the real Silver snapshot
(§6), the oracle-quarantine guard re-asserted at the RDF layer and passing
(§6a), the fluid classification and three directional guards run against
real connectivity (§6b), the SPARQL query surface (§7), and an illustrative
Fuseki push (§8).

**A real-data finding from that first run, now handled, not worked around:**
`silver_cdc`'s `anchor` is a bucket key for components -- two same-class
siblings on one segment can share it (`silver_layer_spec.md` §3.5, the same
non-uniqueness §3f already documents for `seg_tag`), so a strict
`apply_delta` call can hit a real "NEW delta for anchor already current"
collision. §5 now applies events through `apply_silver_cdc_events_tolerant`
(`gold/silver_cdc.py`), which flags each such collision as a `CdcAnomaly`
instead of aborting the batch -- this project's own Stage-D "observe and
record, don't crash on dirty data" discipline. 60/60 tests still green,
3 new ones covering this path (`tests/test_silver_cdc.py`).

**Known gaps, left honest rather than papered over:** `map_connection`
addresses every endpoint as a component (Nozzle-typed edges need a real
`from_kind`/`to_kind` before this feeds a systemization run); the
`Reference_Data.xlsx` column renames in §6 are guesses -- check the printed
`.columns` output; §5b's main-walkthrough Stage E call needs a second
ingested Bronze revision to do anything; `graph:results` (computed
commissioning systems) isn't populated by anything in this notebook yet --
that's the walk.py global fragment-partition step the strategy keeps in
Python (`medallion_rdf_ido_strategy_mapping.md` §6), not shown here.

See `claude/gold_layer_spec.md` for the full design rationale, open risks
(§10), and `gold_layer/medallion_concepts.ipynb` (delivered alongside this
notebook) for the same Gold functionality exercised standalone against a
small synthetic fixture, with every cell actually executed.
